# KV Cache Implementation for LLM Inference

**Stage 1: Basic Optimization**

This notebook demonstrates Key-Value (KV) caching, the single most important optimization for LLM inference. KV caching can provide **5-10x speedup** for text generation.

## What is KV Caching?

In transformer models, during autoregressive generation:
- Each token generation requires attention over ALL previous tokens
- Without caching: we recompute K and V matrices for all previous tokens at each step
- With caching: we store K and V from previous steps and only compute for the new token

### Why KV Caching is Critical

1. **Computational Savings**: Avoid O(n²) recomputation
2. **Memory-Speed Trade-off**: Use more memory to gain massive speed
3. **Essential for Production**: Without it, inference is prohibitively slow

### Memory Cost Formula

```
KV Cache Size = 2 × batch_size × num_layers × num_heads × head_dim × seq_len × bytes_per_param
```

For a 7B parameter model (Llama 2):
- 32 layers, 32 heads, head_dim=128
- For 2048 tokens in FP16: ~1GB per batch item

**Reference**: See `LLM_INFERENCE_ROADMAP.md` - Stage 1, Basic Optimizations

In [ ]:
# Install required packages
!pip install torch transformers accelerate matplotlib pandas -q

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import matplotlib.pyplot as plt
import pandas as pd
from typing import Optional, Tuple
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Manual KV Cache Implementation

Let's implement KV caching from scratch to understand the mechanism.

In [ ]:
class SimpleAttentionWithCache(nn.Module):
    """Simplified attention mechanism with KV caching."""
    
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        # Q, K, V projection layers
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.o_proj = nn.Linear(hidden_size, hidden_size)
        
    def forward(
        self,
        hidden_states: torch.Tensor,
        past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Args:
            hidden_states: (batch_size, seq_len, hidden_size)
            past_key_value: Cached (key, value) from previous steps
            use_cache: Whether to return cache for next step
        
        Returns:
            output: Attention output
            cache: Updated (key, value) cache if use_cache=True
        """
        batch_size, seq_len, _ = hidden_states.shape
        
        # Project to Q, K, V
        query = self.q_proj(hidden_states)
        key = self.k_proj(hidden_states)
        value = self.v_proj(hidden_states)
        
        # Reshape for multi-head attention
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = key.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = value.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply KV cache
        if past_key_value is not None:
            past_key, past_value = past_key_value
            key = torch.cat([past_key, key], dim=2)  # Concatenate along sequence dimension
            value = torch.cat([past_value, value], dim=2)
        
        # Store cache for next iteration
        if use_cache:
            cache = (key, value)
        else:
            cache = None
        
        # Compute attention scores
        scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, value)
        
        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.hidden_size)
        output = self.o_proj(attn_output)
        
        return output, cache


# Test the implementation
attention = SimpleAttentionWithCache(hidden_size=768, num_heads=12).to(device)

# Simulate generation: first token (prompt), then 5 generated tokens
print("Testing KV Cache Implementation:")
print("=" * 50)

# First forward pass (prompt processing)
hidden = torch.randn(1, 10, 768).to(device)  # Batch=1, prompt_len=10
output, cache = attention(hidden, past_key_value=None, use_cache=True)
print(f"Initial pass - Input shape: {hidden.shape}")
print(f"Initial pass - Output shape: {output.shape}")
print(f"Initial pass - Cache K shape: {cache[0].shape}, V shape: {cache[1].shape}")
print()

# Generate 5 tokens using cache
for i in range(5):
    hidden = torch.randn(1, 1, 768).to(device)  # Single new token
    output, cache = attention(hidden, past_key_value=cache, use_cache=True)
    print(f"Token {i+1} - Input shape: {hidden.shape}, Cache size: {cache[0].shape[2]} tokens")

## 2. Memory vs Speed Trade-off Analysis

In [ ]:
def calculate_kv_cache_memory(
    batch_size: int,
    num_layers: int,
    num_heads: int,
    head_dim: int,
    seq_len: int,
    dtype_bytes: int = 2  # FP16
) -> float:
    """Calculate KV cache memory in GB."""
    # Factor of 2 for both K and V
    memory_bytes = 2 * batch_size * num_layers * num_heads * head_dim * seq_len * dtype_bytes
    return memory_bytes / (1024 ** 3)  # Convert to GB


# Example: Llama 2 7B configuration
configs = {
    "Llama 2 7B": {"num_layers": 32, "num_heads": 32, "head_dim": 128},
    "GPT-2 Small": {"num_layers": 12, "num_heads": 12, "head_dim": 64},
    "GPT-2 Medium": {"num_layers": 24, "num_heads": 16, "head_dim": 64},
}

sequence_lengths = [512, 1024, 2048, 4096]
batch_sizes = [1, 4, 8, 16]

print("KV Cache Memory Requirements (FP16)")
print("=" * 80)

for model_name, config in configs.items():
    print(f"\n{model_name}:")
    print(f"  Layers: {config['num_layers']}, Heads: {config['num_heads']}, Head Dim: {config['head_dim']}")
    print(f"\n  {'Seq Len':<10} {'Batch=1':<12} {'Batch=4':<12} {'Batch=8':<12} {'Batch=16':<12}")
    print("  " + "-" * 58)
    
    for seq_len in sequence_lengths:
        row = f"  {seq_len:<10}"
        for batch_size in batch_sizes:
            memory_gb = calculate_kv_cache_memory(
                batch_size, config['num_layers'], config['num_heads'], 
                config['head_dim'], seq_len
            )
            row += f" {memory_gb:>6.2f} GB   "
        print(row)

In [ ]:
# Visualize memory requirements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Memory vs Sequence Length
for model_name, config in configs.items():
    memory_per_seq = [
        calculate_kv_cache_memory(1, config['num_layers'], config['num_heads'], 
                                 config['head_dim'], seq_len)
        for seq_len in sequence_lengths
    ]
    axes[0].plot(sequence_lengths, memory_per_seq, marker='o', label=model_name)

axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Memory (GB)')
axes[0].set_title('KV Cache Memory vs Sequence Length (Batch=1, FP16)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Memory vs Batch Size
seq_len = 2048
for model_name, config in configs.items():
    memory_per_batch = [
        calculate_kv_cache_memory(bs, config['num_layers'], config['num_heads'], 
                                 config['head_dim'], seq_len)
        for bs in batch_sizes
    ]
    axes[1].plot(batch_sizes, memory_per_batch, marker='s', label=model_name)

axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Memory (GB)')
axes[1].set_title(f'KV Cache Memory vs Batch Size (Seq Len={seq_len}, FP16)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kv_cache_memory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insights:")
print("1. Memory scales linearly with sequence length and batch size")
print("2. Larger models need significantly more cache memory")
print("3. FP16 uses 2 bytes per parameter (vs 4 for FP32)")
print("4. For production: balance batch size with available GPU memory")

## 3. Benchmarking: With vs Without KV Cache

Let's measure the actual speedup on a real model (GPT-2).

In [ ]:
# Load GPT-2 model
model_name = "gpt2"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
def generate_without_cache(model, input_ids, max_new_tokens=50):
    """Generate tokens WITHOUT KV caching (very slow)."""
    generated_ids = input_ids.clone()
    
    for _ in range(max_new_tokens):
        # Process entire sequence every time (no caching)
        with torch.no_grad():
            outputs = model(generated_ids, use_cache=False)
            logits = outputs.logits
        
        # Get next token
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        
        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return generated_ids


def generate_with_cache(model, input_ids, max_new_tokens=50):
    """Generate tokens WITH KV caching (fast)."""
    generated_ids = input_ids.clone()
    past_key_values = None
    
    for i in range(max_new_tokens):
        with torch.no_grad():
            if i == 0:
                # First pass: process entire prompt
                outputs = model(generated_ids, use_cache=True)
            else:
                # Subsequent passes: only process new token with cache
                outputs = model(
                    generated_ids[:, -1:], 
                    past_key_values=past_key_values,
                    use_cache=True
                )
            
            logits = outputs.logits
            past_key_values = outputs.past_key_values
        
        # Get next token
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        
        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return generated_ids


# Test prompts of varying lengths
test_prompts = [
    "The quick brown fox",
    "Artificial intelligence is transforming",
    "In a world where technology advances rapidly, we must consider",
    "The history of computing began with mechanical calculators and evolved through " + 
    "vacuum tubes, transistors, and integrated circuits to",
]

print("Running benchmarks...\n")

In [ ]:
results = []
num_tokens_to_generate = 50

for idx, prompt in enumerate(test_prompts):
    print(f"\nPrompt {idx+1}: '{prompt[:50]}...'" if len(prompt) > 50 else f"\nPrompt {idx+1}: '{prompt}'")
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    prompt_length = input_ids.shape[1]
    
    # Warmup
    _ = generate_with_cache(model, input_ids, max_new_tokens=5)
    
    # Benchmark WITHOUT cache
    torch.cuda.synchronize() if device == "cuda" else None
    start = time.time()
    output_no_cache = generate_without_cache(model, input_ids, max_new_tokens=num_tokens_to_generate)
    torch.cuda.synchronize() if device == "cuda" else None
    time_no_cache = time.time() - start
    
    # Benchmark WITH cache
    torch.cuda.synchronize() if device == "cuda" else None
    start = time.time()
    output_with_cache = generate_with_cache(model, input_ids, max_new_tokens=num_tokens_to_generate)
    torch.cuda.synchronize() if device == "cuda" else None
    time_with_cache = time.time() - start
    
    speedup = time_no_cache / time_with_cache
    tokens_per_sec_no_cache = num_tokens_to_generate / time_no_cache
    tokens_per_sec_with_cache = num_tokens_to_generate / time_with_cache
    
    results.append({
        'Prompt Length': prompt_length,
        'Time w/o Cache (s)': time_no_cache,
        'Time w/ Cache (s)': time_with_cache,
        'Speedup': speedup,
        'Tokens/sec (no cache)': tokens_per_sec_no_cache,
        'Tokens/sec (cached)': tokens_per_sec_with_cache,
    })
    
    print(f"  Prompt length: {prompt_length} tokens")
    print(f"  Without cache: {time_no_cache:.3f}s ({tokens_per_sec_no_cache:.1f} tokens/s)")
    print(f"  With cache:    {time_with_cache:.3f}s ({tokens_per_sec_with_cache:.1f} tokens/s)")
    print(f"  Speedup:       {speedup:.2f}x")

# Create results DataFrame
df_results = pd.DataFrame(results)
print("\n" + "="*80)
print("\nBenchmark Results Summary:")
print(df_results.to_string(index=False))
print(f"\nAverage Speedup: {df_results['Speedup'].mean():.2f}x")
print(f"Min Speedup: {df_results['Speedup'].min():.2f}x")
print(f"Max Speedup: {df_results['Speedup'].max():.2f}x")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Time comparison
x = np.arange(len(df_results))
width = 0.35
axes[0].bar(x - width/2, df_results['Time w/o Cache (s)'], width, label='Without Cache', color='coral')
axes[0].bar(x + width/2, df_results['Time w/ Cache (s)'], width, label='With Cache', color='lightgreen')
axes[0].set_xlabel('Prompt Index')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Generation Time: With vs Without KV Cache')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"P{i+1}\n({df_results['Prompt Length'].iloc[i]})" for i in range(len(df_results))])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Speedup vs Prompt Length
axes[1].plot(df_results['Prompt Length'], df_results['Speedup'], 
             marker='o', linewidth=2, markersize=8, color='darkblue')
axes[1].axhline(y=1, color='red', linestyle='--', label='No speedup', alpha=0.5)
axes[1].set_xlabel('Prompt Length (tokens)')
axes[1].set_ylabel('Speedup (x)')
axes[1].set_title('KV Cache Speedup vs Prompt Length')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Annotate speedup values
for i, row in df_results.iterrows():
    axes[1].annotate(f"{row['Speedup']:.1f}x", 
                    (row['Prompt Length'], row['Speedup']),
                    textcoords="offset points", xytext=(0,10), ha='center')

plt.tight_layout()
plt.savefig('kv_cache_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Findings:")
print(f"1. KV caching provides {df_results['Speedup'].mean():.1f}x average speedup")
print(f"2. Speedup increases with longer prompts (more computation to save)")
print(f"3. Throughput improved from {df_results['Tokens/sec (no cache)'].mean():.1f} to {df_results['Tokens/sec (cached)'].mean():.1f} tokens/sec")
print("4. This is THE most critical optimization for production LLM inference")

## 4. Integration with HuggingFace Models

HuggingFace Transformers has built-in KV cache support. Here's how to use it effectively.

In [ ]:
# Using HuggingFace's built-in KV cache (via generate())
prompt = "The future of artificial intelligence includes"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print("HuggingFace generate() automatically uses KV caching:\n")

# Method 1: Standard generate (uses cache by default)
start = time.time()
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=False,
        use_cache=True,  # Default: True
    )
time_cached = time.time() - start

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated text: {generated_text}")
print(f"\nGeneration time (with cache): {time_cached:.3f}s")
print(f"Tokens per second: {50 / time_cached:.1f}")

# Method 2: Explicitly disable cache (not recommended!)
start = time.time()
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=False,
        use_cache=False,  # Explicitly disabled
    )
time_no_cache = time.time() - start

print(f"\nGeneration time (without cache): {time_no_cache:.3f}s")
print(f"Tokens per second: {50 / time_no_cache:.1f}")
print(f"\nSpeedup with cache: {time_no_cache / time_cached:.2f}x")

print("\n" + "="*80)
print("Production Best Practices:")
print("1. ALWAYS use use_cache=True (it's the default)")
print("2. Monitor GPU memory usage - cache can be large")
print("3. For batch inference, cache grows linearly with batch size")
print("4. Consider cache memory when setting max_length limits")

## 5. Advanced: Monitoring Cache Memory Usage

In [ ]:
def measure_cache_memory(model, input_ids, max_new_tokens=100):
    """Measure actual KV cache memory usage."""
    if device == "cpu":
        print("GPU memory measurement only available on CUDA devices")
        return
    
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    mem_before = torch.cuda.memory_allocated() / (1024 ** 3)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            use_cache=True,
        )
    
    mem_after = torch.cuda.memory_allocated() / (1024 ** 3)
    mem_peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    return {
        'before': mem_before,
        'after': mem_after,
        'peak': mem_peak,
        'cache_size': mem_after - mem_before
    }


if device == "cuda":
    print("Measuring cache memory for different generation lengths:\n")
    
    prompt = "The history of computing began with"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    generation_lengths = [50, 100, 200, 500]
    memory_results = []
    
    for gen_len in generation_lengths:
        mem_info = measure_cache_memory(model, input_ids, max_new_tokens=gen_len)
        memory_results.append({
            'Generation Length': gen_len,
            'Peak Memory (GB)': mem_info['peak'],
            'Cache Size (GB)': mem_info['cache_size'],
        })
        print(f"Tokens: {gen_len:3d} | Peak: {mem_info['peak']:.3f} GB | Cache: {mem_info['cache_size']:.3f} GB")
    
    df_mem = pd.DataFrame(memory_results)
    
    # Plot memory usage
    plt.figure(figsize=(10, 6))
    plt.plot(df_mem['Generation Length'], df_mem['Cache Size (GB)'], 
             marker='o', linewidth=2, markersize=8)
    plt.xlabel('Number of Generated Tokens')
    plt.ylabel('KV Cache Memory (GB)')
    plt.title('KV Cache Memory vs Generation Length')
    plt.grid(True, alpha=0.3)
    plt.savefig('cache_memory_usage.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nLinear growth confirmed: {df_mem['Cache Size (GB)'].iloc[-1] / df_mem['Cache Size (GB)'].iloc[0]:.1f}x " +
          f"memory for {generation_lengths[-1] / generation_lengths[0]:.1f}x tokens")
else:
    print("Skipping GPU memory analysis (CPU mode)")

## 6. Production Deployment Checklist

In [ ]:
def production_kv_cache_config(model_config: dict) -> dict:
    """
    Calculate optimal KV cache configuration for production.
    
    Args:
        model_config: Dict with 'num_layers', 'num_heads', 'head_dim'
    
    Returns:
        Recommended configuration dict
    """
    recommendations = {
        'always_use_cache': True,
        'monitor_memory': True,
    }
    
    # Calculate memory per token per batch item (in GB)
    memory_per_token_gb = calculate_kv_cache_memory(
        batch_size=1,
        num_layers=model_config['num_layers'],
        num_heads=model_config['num_heads'],
        head_dim=model_config['head_dim'],
        seq_len=1,
        dtype_bytes=2  # FP16
    )
    
    recommendations['memory_per_token_gb'] = memory_per_token_gb
    
    # Example GPU memory limits
    gpu_memory_gb = 16  # Example: RTX 4080
    model_memory_gb = 2  # Example: GPT-2 in FP16
    available_for_cache = gpu_memory_gb - model_memory_gb - 2  # 2GB buffer
    
    max_tokens_per_batch = int(available_for_cache / memory_per_token_gb)
    
    recommendations['max_tokens_per_batch'] = max_tokens_per_batch
    recommendations['suggested_max_length'] = min(max_tokens_per_batch, 2048)
    recommendations['suggested_batch_size'] = max(1, available_for_cache // (memory_per_token_gb * 512))
    
    return recommendations


# Example for GPT-2
gpt2_config = {'num_layers': 12, 'num_heads': 12, 'head_dim': 64}
config = production_kv_cache_config(gpt2_config)

print("Production KV Cache Configuration")
print("="*80)
print(f"Model: GPT-2 (124M parameters)")
print(f"\nMemory per token (batch=1): {config['memory_per_token_gb']*1024:.2f} MB")
print(f"Maximum tokens per batch: {config['max_tokens_per_batch']:,}")
print(f"Suggested max_length: {config['suggested_max_length']}")
print(f"Suggested batch_size (for 512 token sequences): {config['suggested_batch_size']}")
print(f"\nAlways use cache: {config['always_use_cache']}")

print("\n" + "="*80)
print("\nProduction Checklist:")
checklist = [
    "✓ Enable use_cache=True in model.generate()",
    "✓ Calculate KV cache memory for your model size",
    "✓ Set appropriate max_length based on available GPU memory",
    "✓ Monitor GPU memory usage in production",
    "✓ Adjust batch size to stay within memory limits",
    "✓ Consider FP16 to halve cache memory requirements",
    "✓ Profile inference latency with different sequence lengths",
    "✓ Set up alerts for OOM errors",
]

for item in checklist:
    print(f"  {item}")

## Summary and Key Takeaways

### Performance Results
- **Speedup**: 5-10x faster generation (varies with prompt length)
- **Throughput**: 10-50+ tokens/sec depending on model and hardware
- **Cost**: Linear memory growth with sequence length

### Memory Formulas
```
KV Cache (bytes) = 2 × batch × layers × heads × head_dim × seq_len × dtype_bytes
KV Cache (GB) = KV Cache (bytes) / (1024³)
```

### When KV Caching is Most Effective
1. **Long sequence generation**: More recomputation saved
2. **Autoregressive tasks**: Each token depends on previous
3. **Interactive applications**: Conversational AI, code completion
4. **Batch inference**: Amortize overhead across multiple requests

### Production Recommendations
1. Always enable KV caching (it's default in HuggingFace)
2. Calculate cache memory requirements for your model
3. Set max_length based on available GPU memory
4. Monitor memory usage in production
5. Combine with FP16 for 2x memory savings

### Next Steps
- **Stage 1**: Mixed Precision (Notebook 11) - 2x memory reduction
- **Stage 1**: Static Batching (Notebook 12) - Higher throughput
- **Stage 2**: PagedAttention - More efficient cache management

### References
- LLM_INFERENCE_ROADMAP.md - Stage 1: Basic Optimizations
- HuggingFace Transformers: https://huggingface.co/docs/transformers/main_classes/text_generation
- Attention is All You Need: https://arxiv.org/abs/1706.03762